# Random Forests — Inventing Ensemble Learning

In the **Decision Trees** chapter you invented a tree that predicts *categories*
(male/female, yes/no) by minimizing impurity. Two questions remain open:

1. What if the thing you want to predict is a **number** — a house price, a
   salary, a temperature?
2. A single tree happily *memorizes* its training data. Can we do better?

In this chapter you will invent the answers: first a **regression tree**, then
the idea that powers many production ML systems to this day — **combine many
imperfect trees into one strong model**. That is a Random Forest.

Everything is plain Python. You'll reuse ideas (mean, variance) you built in
**Loops and Arrays**.

---
# Part 1 — Impurity for Numbers

The classification tree asked: "how *mixed* are the labels in this group?"
For numbers the question becomes: "how *spread out* are the values?"

Look at these two groups of house prices (in lakhs):

```
group_a = [10, 10, 10, 10, 10]     # everyone agrees
group_b = [1, 50, 3, 80]           # wild disagreement
```

If a leaf of your tree contains `group_a`, predicting their mean (10) is
perfect. If it contains `group_b`, predicting the mean (33.5) is terrible for
every single house.

### Exercise 1.1 — Invent a Spread Score

Invent a single number that is `0` when all values are identical and grows as
the values spread out. Try to come up with your own formula before reading on.

The classic answer is **variance**: the average *squared* distance from the
mean. (Why squared? So that being 10 below the mean can't cancel out being 10
above it — and big misses get punished extra hard.)

Write `variance(values)`.

**Write your solution in `exercises/ex_1_1_variance.py`**

In [ ]:
def variance(values):
    pass  # replace this


assert variance([10, 10, 10, 10, 10]) == 0
assert variance([1, 50, 3, 80]) == 1105.25
assert variance([1, 3]) == 1.0
assert variance([4]) == 0
print("variance: OK")

**Think:** unlike the 0-to-1 impurity you built for classification, variance
has no upper bound. You *could* squash it into 0–1, but you would lose
information — and a tree never needs the absolute number anyway, it only ever
*compares* one split against another. So raw variance is fine.

### Exercise 1.2 — The Value of a Split

Take this group: `values = [10, 9, 11, 8, 10, 1]`. It has one troublemaker.

Split it into `left = [10, 9, 11, 8, 10]` and `right = [1]`. Now `left` is
nearly pure and `right` is *exactly* pure (a single value has zero variance).

But there's a trap: `[1]` having zero variance shouldn't count as much as a
five-element group having zero variance — otherwise the tree learns to slice
off one element at a time. The fix: weight each side's variance by the
**fraction of elements** it holds.

Write `total_variance(left, right)`:

$$\text{total} = \frac{n_{left}}{n}\,\text{var}(left) + \frac{n_{right}}{n}\,\text{var}(right)$$

**Write your solution in `exercises/ex_1_2_total_variance.py`**

In [ ]:
def total_variance(left, right):
    pass  # replace this


assert total_variance([10, 10, 10], [0, 0, 0]) == 0.0
assert abs(total_variance([10, 9, 11, 8, 10], [1]) - 0.8667) < 1e-3
assert total_variance([5], [5]) == 0.0
print("total_variance: OK")

### Exercise 1.3 — Variance Reduction

A split is good when the children are much purer than the parent. Capture that
in one number:

$$\text{reduction} = \text{var}(parent) - \text{total\_variance}(left, right)$$

Write `variance_reduction(parent, left, right)`. Then check: for
`[10, 9, 11, 8, 10, 1]`, splitting the troublemaker off should score much
higher than splitting down the middle.

**Write your solution in `exercises/ex_1_3_variance_reduction.py`**

In [ ]:
def variance_reduction(parent, left, right):
    pass  # replace this


values = [10, 9, 11, 8, 10, 1]
assert abs(variance(values) - 11.1389) < 1e-3

good = variance_reduction(values, [10, 9, 11, 8, 10], [1])
bad = variance_reduction(values, [10, 9, 11], [8, 10, 1])
print("good split:", round(good, 4), " bad split:", round(bad, 4))
assert abs(good - 10.2722) < 1e-3
assert good > bad
print("variance_reduction: OK")

---
# Part 2 — Choosing the Best Split

Here is a tiny housing dataset. Each row is `[size_sqft, bedrooms]`; the
target is the price:

| Size | Bedrooms | Price |
|------|----------|-------|
| 500  | 1 | 50  |
| 600  | 1 | 55  |
| 900  | 3 | 65  |
| 1500 | 3 | 150 |
| 1600 | 3 | 160 |
| 1700 | 4 | 170 |

### Exercise 2.1 — Compare Two Splits by Hand

Using your `variance_reduction`, compute the reduction for two candidate
splits, and see which feature the tree should ask about first:

- **Size ≤ 1000**: prices split into `[50, 55, 65]` vs `[150, 160, 170]`
- **Bedrooms < 3**: prices split into `[50, 55]` vs `[65, 150, 160, 170]`

In [ ]:
X = [[500, 1], [600, 1], [900, 3], [1500, 3], [1600, 3], [1700, 4]]
y = [50, 55, 65, 150, 160, 170]

size_red = None      # replace this
bedroom_red = None   # replace this

print("size:", round(size_red, 2), " bedrooms:", round(bedroom_red, 2))
assert abs(size_red - 2669.44) < 0.1
assert abs(bedroom_red - 1558.68) < 0.1
assert size_red > bedroom_red
print("Size wins — ask about size first")

### Exercise 2.2 — Find the Best Split Automatically

Trying splits by hand doesn't scale. Write
`find_best_split(X, y, feature_indices=None)` that:

1. Considers every feature index (or only `feature_indices` if given — you'll
   see why that parameter matters in Part 4).
2. For each feature, takes the **midpoints between consecutive sorted unique
   values** as candidate thresholds (e.g. sizes `500, 600, 900, ...` give
   thresholds `550, 750, 1200, ...`).
3. For each candidate `t`, splits the targets into rows with
   `x[feature] <= t` (left) and the rest (right), and computes the variance
   reduction.
4. Returns the best `(feature, threshold, reduction)` — keep the *first*
   candidate when there's a tie.

If there is nothing to split (no candidates), return `(None, None, 0.0)`.

**Write your solution in `exercises/ex_2_2_find_best_split.py`**

In [ ]:
def find_best_split(X, y, feature_indices=None):
    pass  # replace this


f, t, red = find_best_split(X, y)
print("best split: feature", f, "threshold", t, "reduction", round(red, 2))
assert f == 0
assert abs(t - 1200) < 1e-9
assert abs(red - 2669.44) < 0.1
assert find_best_split([[1, 1]], [10]) == (None, None, 0.0)
print("find_best_split: OK")

### When Should the Tree Stop Asking Questions?

Left alone, the tree splits until every leaf holds a single house — that's not
learning, it's **memorization**: perfect on training data, brittle on anything
new. Two brakes prevent it:

- **`max_depth`** — the maximum number of questions from root to leaf. A
  shallow tree is forced to generalize.
- **`min_samples_split`** — the smallest group still allowed to be split.
  Splitting a 2-house group into two 1-house "groups" scores a perfect
  variance reduction, and learns nothing.

You'll wire both into the tree next. There's nothing to compute here — just
convince yourself *why* each brake prevents memorization.

---
# Part 3 — Build the Regression Tree

### The Node — Provided, run this cell as-is

A tree is made of nodes. A **decision node** holds a question
(`feature`, `threshold`) and two children; a **leaf** holds only a `value`
(the prediction: the mean of the targets that reached it).

In [ ]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

    def is_leaf(self):
        return self.value is not None


def mean(values):
    return sum(values) / len(values)


def root_mean_squared_error(y_true, y_pred):
    return (sum((a - b) ** 2 for a, b in zip(y_true, y_pred)) / len(y_true)) ** 0.5


leaf = Node(value=52.5)
assert leaf.is_leaf()
assert not Node(feature=0, threshold=1200, left=leaf, right=leaf).is_leaf()
print("Node ready")

### Exercise 3.1 — The Tree

Now assemble everything into a `DecisionTreeRegressor` class:

```python
class DecisionTreeRegressor:
    def __init__(self, max_depth=5, min_samples_split=2, feature_indices=None):
        ...
    def fit(self, X, y):
        ...
    def predict(self, X):
        ...
```

**`fit`** grows the tree recursively, starting at depth 0. A group becomes a
**leaf** (value = mean of its targets) when *any* brake fires:

- depth has reached `max_depth`, or
- the group has fewer than `min_samples_split` rows, or
- `find_best_split` finds no split with reduction > 0.

Otherwise: split the rows on the best `(feature, threshold)` and recurse into
each side with `depth + 1`. Pass `feature_indices` through to
`find_best_split`.

**`predict`** walks each row from the root — go left if
`row[feature] <= threshold` — and returns the leaf's value.

**Write your solution in `exercises/ex_3_1_decisiontreeregressor.py`**

In [ ]:
class DecisionTreeRegressor:
    pass  # replace this


# A deep tree memorizes this tiny dataset perfectly...
deep = DecisionTreeRegressor(max_depth=5, min_samples_split=2)
deep.fit(X, y)
assert root_mean_squared_error(y, deep.predict(X)) < 1e-9

# ...a depth-1 "stump" can only ask one question, so it must generalize:
stump = DecisionTreeRegressor(max_depth=1, min_samples_split=2)
stump.fit(X, y)
assert root_mean_squared_error(y, stump.predict(X)) > 5

p = deep.predict([[550, 1], [2000, 5]])
print("predictions for a 550 sqft and a 2000 sqft house:", p)
assert 50 <= p[0] <= 65
assert 150 <= p[1] <= 170
print("DecisionTreeRegressor: OK")

---
# Part 4 — Many Trees Are Better Than One

Your deep tree scored a *perfect* 0.00 RMSE on its training data. That should
make you suspicious, not proud — it memorized.

Here's the big idea: train **many** trees and **average** their predictions.
Individual mistakes point in different directions and cancel out; the shared
signal survives. (The same reason a crowd's average guess of a jar's jellybean
count beats almost every individual guess.)

**But wait.** If every tree sees exactly the same data, every tree is
*identical*:

```
tree1 predicts 50,  tree2 predicts 50,  tree3 predicts 50 ... average: 50
```

Averaging identical trees gives you... one tree. To benefit from a crowd, the
trees must be **different**. Random Forests inject randomness in two places.

### Exercise 4.1 — Bootstrap Sampling (randomize the *rows*)

Give each tree its own dataset by sampling `n` rows from the original `n`
rows **with replacement** — some rows appear twice, some not at all. This is
called a **bootstrap sample**.

Write `bootstrap_sample(X, y)` that returns a new `(X_sample, y_sample)` of
the same length, where each position holds a uniformly random row of the
original data — and each `y` stays paired with its `X` row. Use
`random.randint`.

**Write your solution in `exercises/ex_4_1_bootstrap_sample.py`**

In [ ]:
import random

def bootstrap_sample(X, y):
    pass  # replace this


X16 = [[500, 1], [550, 1], [600, 1], [650, 2], [700, 2], [800, 2], [900, 2],
       [1200, 3], [1300, 3], [1400, 3], [1500, 3], [1600, 3],
       [1700, 4], [1800, 4], [1900, 4], [2000, 4]]
y16 = [48, 54, 57, 64, 66, 95, 81, 120, 127, 136, 149, 157, 171, 205, 189, 197]

random.seed(7)
Xs, ys = bootstrap_sample(X16, y16)
assert len(Xs) == len(X16) and len(ys) == len(y16)
assert all(row in X16 for row in Xs)
assert all(ys[i] == y16[X16.index(Xs[i])] for i in range(len(Xs)))
assert Xs != X16    # drawing the original order back is astronomically unlikely
print("bootstrap_sample: OK")

### Exercise 4.2 — Feature Randomness (randomize the *questions*)

Bootstrap samples still look similar, so most trees would still put the
strongest feature (here: size) at the root — the crowd stays too agreeable.
Second injection of randomness: let each tree see only a **random subset of
the features**.

Write `choose_features(n_features, max_features)` that returns
`max_features` *distinct* feature indices chosen at random from
`0 .. n_features - 1`. (`random.sample` is your friend.)

This is why `find_best_split` takes that `feature_indices` parameter.

**Write your solution in `exercises/ex_4_2_choose_features.py`**

In [ ]:
def choose_features(n_features, max_features):
    pass  # replace this


feats = choose_features(4, 2)
print("this tree may only ask about features:", feats)
assert len(feats) == 2
assert len(set(feats)) == 2
assert set(feats) <= {0, 1, 2, 3}
print("choose_features: OK")

---
# Part 5 — The Random Forest

### Exercise 5.1 — Assemble It

Write `RandomForestRegressor`:

```python
class RandomForestRegressor:
    def __init__(self, n_estimators=10, max_depth=3,
                 min_samples_split=2, max_features=None):
        ...
    def fit(self, X, y):
        ...
    def predict(self, X):
        ...
```

**`fit`**: for each of `n_estimators` trees — take a bootstrap sample, choose
a random feature subset (all features if `max_features` is `None`), train a
`DecisionTreeRegressor` on the sample with those `feature_indices`, and store
it in `self.trees`.

**`predict`**: ask every tree, and return the **average** of their answers
for each row.

**Write your solution in `exercises/ex_5_1_randomforestregressor.py`**

In [ ]:
class RandomForestRegressor:
    pass  # replace this


X_test = [[575, 1], [750, 2], [1350, 3], [1650, 3], [1950, 4]]
y_test = [55, 70, 132, 165, 192]

random.seed(42)
rf = RandomForestRegressor(n_estimators=20, max_depth=3,
                           min_samples_split=2, max_features=1)
rf.fit(X16, y16)
assert len(rf.trees) == 20

rf_train = root_mean_squared_error(y16, rf.predict(X16))
rf_test = root_mean_squared_error(y_test, rf.predict(X_test))

single = DecisionTreeRegressor(max_depth=10, min_samples_split=2)
single.fit(X16, y16)
single_train = root_mean_squared_error(y16, single.predict(X16))
single_test = root_mean_squared_error(y_test, single.predict(X_test))

print(f"single tree — train RMSE {single_train:6.2f}   test RMSE {single_test:6.2f}")
print(f"forest (20) — train RMSE {rf_train:6.2f}   test RMSE {rf_test:6.2f}")
assert rf_train < 30 and rf_test < 30
print("RandomForestRegressor: OK")

### Exercise 5.2 — Think: When Does the Crowd Win?

Look at your numbers. The single deep tree hit **0.00 train RMSE**
(memorization) — yet on this tiny, perfectly clean dataset its *test* error
may still beat the forest's. So was all this for nothing?

No — but the win shows up under *real* conditions. Reason through each:

1. Add measurement noise to the prices (e.g. `y + random.gauss(0, 15)` per
   row). Which model's test error degrades more, and why? (Try it!)
2. The single tree changes drastically if you delete one training row — the
   forest barely moves. Why does averaging make predictions *stable*?
3. Why is `max_features` more valuable when you have 50 features than when
   you have 2?

---
## Summary

| Idea | You built | Why it matters |
|------|-----------|----------------|
| Variance as impurity | `variance`, `total_variance`, `variance_reduction` | extends decision trees from categories to numbers |
| Best split | `find_best_split` | the greedy heart of every tree learner |
| Regression tree | `Node`, `DecisionTreeRegressor` | `fit`/`predict` with `max_depth`, `min_samples_split` brakes |
| Bootstrap + feature randomness | `bootstrap_sample`, `choose_features` | makes trees *disagree* so averaging helps |
| Ensemble | `RandomForestRegressor` | many weak models → one strong, stable model |

You have now invented, from scratch, the algorithm behind scikit-learn's
`RandomForestRegressor`. The next big idea in tree ensembles — training each
new tree on the *errors* of the previous ones — is called **gradient
boosting** (XGBoost, LightGBM). You already own every ingredient it needs.